In [1]:
# 1. Install required library for this Colab session
!pip install nilearn

import os
import numpy as np
import pandas as pd
from google.colab import drive
from nilearn import datasets
from nilearn.connectome import ConnectivityMeasure

# 2. Mount Drive (Essential for accessing the raw data we just downloaded)
drive.mount('/content/drive')

# 3. Define Paths using your updated directory name
base_dir = '/content/drive/MyDrive/ASD_GNN_Research2'
raw_data_path = os.path.join(base_dir, 'raw_data')
ts_dir = os.path.join(base_dir, 'preprocessed', 'time_series')
conn_dir = os.path.join(base_dir, 'connectivity')

# 4. Fetch/Load Data from your Drive Cache
print("Loading ABIDE dataset from Drive...")
abide_data = datasets.fetch_abide_pcp(
    data_dir=raw_data_path,
    pipeline='cpac',
    derivatives=['rois_aal'],
    quality_checked=True
)

pheno_df = pd.DataFrame(abide_data.phenotypic)
pheno_df['DX_GROUP'] = pheno_df['DX_GROUP'].map({1: 1, 2: 0})

valid_subjects = []
time_series_list = []

# 5. Process and Save Time-Series Data (Nodes)
print("Extracting time-series arrays and standardizing shapes...")
for idx, sub_id in enumerate(pheno_df['SUB_ID']):
    ts_data = abide_data.rois_aal[idx]

    if isinstance(ts_data, str):
        ts_array = np.loadtxt(ts_data)
    else:
        ts_array = ts_data

    # Strict Quality Control: Ensure the matrix has exactly 116 regions
    if len(ts_array.shape) > 1 and ts_array.shape[1] == 116:
        np.save(os.path.join(ts_dir, f'sub_{sub_id}_ts.npy'), ts_array)
        valid_subjects.append(sub_id)
        time_series_list.append(ts_array)

print(f"Saved clean time-series for {len(valid_subjects)} subjects.")

# Save the Cleaned Labels CSV
pheno_clean = pheno_df[pheno_df['SUB_ID'].isin(valid_subjects)]
pheno_clean.to_csv(os.path.join(raw_data_path, 'phenotypic_cleaned.csv'), index=False)

# 6. Compute and Save Connectivity Matrices (Edges)
print("\nComputing Functional Connectivity Matrices (Pearson Correlation)...")
conn_measure = ConnectivityMeasure(kind='correlation')
connectivity_matrices = conn_measure.fit_transform(time_series_list)

for idx, sub_id in enumerate(valid_subjects):
    np.save(os.path.join(conn_dir, f'sub_{sub_id}_conn.npy'), connectivity_matrices[idx])

print("Connectivity matrices successfully saved to Drive!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 2.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
Moun

[fetch_abide_pcp] Dataset directory found: /content/drive/MyDrive/ASD_GNN_Research2/raw_data/ABIDE_pcp

Extracting time-series arrays and standardizing shapes...
Saved clean time-series for 871 subjects.

Computing Functional Connectivity Matrices (Pearson Correlation)...
Connectivity matrices successfully saved to Drive!
